# Autoresearch Experiment Analysis

Analysis of the autonomous research ratchet from `results.tsv` — image geolocalization,
objective **`median_km`** (median great-circle error on val, lower is better).

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np

# Load the TSV (tab-separated, 5 columns: commit, score, memory_gb, status, description)
df = pd.read_csv("results.tsv", sep="\t")
df = df.rename(columns={"score": "median_km"})
df["median_km"] = pd.to_numeric(df["median_km"], errors="coerce")
df["memory_gb"] = pd.to_numeric(df["memory_gb"], errors="coerce")
df["status"] = df["status"].str.strip().str.upper()

# Session boundaries: each session starts with a fresh baseline re-run
df["is_baseline"] = df["description"].str.contains("baseline", case=False, na=False)
df["session"] = df["is_baseline"].cumsum()

print(f"Total experiments: {len(df)} across {df['session'].max()} sessions")
df.head(10)

In [ ]:
counts = df["status"].value_counts()
print("Experiment outcomes:")
print(counts.to_string())

n_keep = counts.get("KEEP", 0)
n_discard = counts.get("DISCARD", 0)
n_crash = counts.get("CRASH", 0)
n_decided = n_keep + n_discard
if n_decided > 0:
    print(f"\nKeep rate: {n_keep}/{n_decided} = {n_keep / n_decided:.1%}")

In [ ]:
# Show all KEPT experiments (the improvements that stuck)
kept = df[df["status"] == "KEEP"].copy()
print(f"KEPT experiments ({len(kept)} total):\n")
for i, row in kept.iterrows():
    print(f"  #{i:3d}  S{row['session']}  median={row['median_km']:7.1f} km  "
          f"mem={row['memory_gb']:.1f}GB  {str(row['description'])[:70]}")

## Median error over time

Champion trajectory across all sessions: every experiment as a dot, the running-best (kept
champion) as a step line, session boundaries marked at each baseline re-run.

In [ ]:
fig, ax = plt.subplots(figsize=(16, 8))

valid = df[(df["status"] != "CRASH") & df["median_km"].notna() & (df["median_km"] > 0)].copy()
valid = valid.reset_index(drop=True)

# Session shading + labels
sess_starts = valid.index[valid["is_baseline"]].tolist() + [len(valid)]
for si in range(len(sess_starts) - 1):
    a, b = sess_starts[si], sess_starts[si + 1]
    if si % 2 == 1:
        ax.axvspan(a - 0.5, b - 0.5, color="#f2f2f2", zorder=0)
    ax.text((a + b - 1) / 2, 0.985, f"session {si + 1}", transform=ax.get_xaxis_transform(),
            ha="center", va="top", fontsize=9, color="#888888")

disc = valid[valid["status"] == "DISCARD"]
ax.scatter(disc.index, disc["median_km"], c="#bbbbbb", s=18, alpha=0.7, zorder=2,
           label="Discarded")

kept_mask = valid["status"] == "KEEP"
kept_idx = valid.index[kept_mask]
kept_km = valid.loc[kept_mask, "median_km"]
ax.scatter(kept_idx, kept_km, c="#2ecc71", s=55, zorder=4, label="Kept",
           edgecolors="black", linewidths=0.5)

running_min = kept_km.cummin()
ax.step(kept_idx, running_min, where="post", color="#27ae60", linewidth=2, alpha=0.8,
        zorder=3, label="Champion (running best)")

# Label only experiments that MOVED the champion (new running best)
new_best = kept_km == running_min
prev_best = running_min.shift(1)
moved = new_best & (prev_best.isna() | (kept_km < prev_best - 0.5))
for idx, km in zip(kept_idx[moved], kept_km[moved]):
    desc = str(valid.loc[idx, "description"]).strip()
    desc = desc.split(":")[0][:44]
    ax.annotate(f"{desc}\n{km:.0f} km", (idx, km), textcoords="offset points",
                xytext=(6, 8), fontsize=8, color="#1a7a3a", rotation=25,
                ha="left", va="bottom")

n_total, n_kept = len(df), int((df["status"] == "KEEP").sum())
best = kept_km.min()
base = valid.loc[0, "median_km"]
ax.set_xlabel("Experiment #", fontsize=12)
ax.set_ylabel("median_km (lower is better)", fontsize=12)
ax.set_title(f"Autoresearch progress: {base:.0f} → {best:.0f} km "
             f"(−{(base - best) / base * 100:.0f}%) over {n_total} experiments, {n_kept} kept",
             fontsize=14)
ax.legend(loc="upper right", fontsize=9)
ax.grid(True, alpha=0.2)
ax.set_ylim(best * 0.82, base * 1.06)

plt.tight_layout()
plt.savefig("progress.png", dpi=150, bbox_inches="tight")
plt.show()
print("Saved to progress.png")

## Summary Statistics

In [ ]:
# Summary stats
kept = df[(df["status"] == "KEEP") & (df["median_km"] > 0)].copy()
baseline_km = df.iloc[1]["median_km"] if df.iloc[0]["median_km"] == 0 else df.iloc[0]["median_km"]
best_km = kept["median_km"].min()
best_row = kept.loc[kept["median_km"].idxmin()]

print(f"Baseline median_km:  {baseline_km:.1f}")
print(f"Best median_km:      {best_km:.1f}")
print(f"Total improvement:   {baseline_km - best_km:.1f} km ({(baseline_km - best_km) / baseline_km * 100:.1f}%)")
print(f"Best experiment:     {best_row['description']}")
print()
print("Champion trajectory (kept experiments):")
for _, row in kept.iterrows():
    print(f"  S{row['session']}  {row['median_km']:7.1f} km  {str(row['description'])[:75]}")

## Top Hits (Kept Experiments by Improvement)

In [ ]:
# Each kept experiment's delta vs the previous kept score (experiments are cumulative)
kept = df[(df["status"] == "KEEP") & (df["median_km"] > 0)].copy()
kept["prev_km"] = kept["median_km"].shift(1)
kept["delta"] = kept["prev_km"] - kept["median_km"]

hits = kept.iloc[1:].sort_values("delta", ascending=False)

print(f"{'Rank':>4}  {'Delta km':>9}  {'median':>8}  Description")
print("-" * 100)
for rank, (_, row) in enumerate(hits.iterrows(), 1):
    print(f"{rank:4d}  {row['delta']:+9.1f}  {row['median_km']:8.1f}  {str(row['description'])[:75]}")

print(f"\n{'':>4}  {hits['delta'].sum():+9.1f}  {'':>8}  TOTAL improvement over baseline")